# Week 2 EDA — Credit Risk News Monitor

Exploratory analysis of the pipeline output after Phase 2 (cleaning, NER, entity mapping).
Sentiment scoring is not yet done (Phase 3), so `sentiment_score` and `avg_sentiment` are all NULL.

In [ ]:
# Cell 1 — Load data
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

engine = create_engine(os.getenv("DATABASE_URL"))

articles_df = pd.read_sql(
    "SELECT id, title, source, published_at, language FROM articles", engine
)
processed_df = pd.read_sql(
    "SELECT article_id, sentiment_score, sentiment_label, is_credit_relevant FROM processed_articles",
    engine,
)
obligors_df = pd.read_sql(
    "SELECT id, name, ticker, sector FROM obligors", engine
)
links_df = pd.read_sql(
    "SELECT article_id, obligor_id FROM article_obligors", engine
)
signals_df = pd.read_sql("SELECT * FROM obligor_daily_signals", engine)

print(f"articles:              {len(articles_df):>6,}")
print(f"processed_articles:    {len(processed_df):>6,}")
print(f"obligors:              {len(obligors_df):>6,}")
print(f"article_obligors:      {len(links_df):>6,}")
print(f"obligor_daily_signals: {len(signals_df):>6,}")

In [ ]:
# Cell 2 — Stats summary
total_articles = len(articles_df)
processed_count = len(processed_df)
coverage_pct = processed_count / total_articles * 100 if total_articles else 0

articles_df["published_at"] = pd.to_datetime(articles_df["published_at"])
date_range = (
    articles_df["published_at"].min().date(),
    articles_df["published_at"].max().date(),
)

avg_sentiment = processed_df["sentiment_score"].mean()
avg_sentiment_str = f"{avg_sentiment:.3f}" if pd.notna(avg_sentiment) else "N/A (FinBERT not yet run)"

print("=== Pipeline Stats ===")
print(f"Total articles:         {total_articles:,}")
print(f"Processed articles:     {processed_count:,}")
print(f"Processing coverage:    {coverage_pct:.1f}%")
print(f"Date range:             {date_range[0]} → {date_range[1]}")
print(f"Avg sentiment score:    {avg_sentiment_str}")

In [ ]:
# Cell 3 — Obligor coverage
import matplotlib.pyplot as plt

article_counts = (
    links_df.groupby("obligor_id")["article_id"]
    .count()
    .reset_index()
    .rename(columns={"article_id": "article_count"})
    .merge(obligors_df[["id", "name", "ticker"]], left_on="obligor_id", right_on="id")
    .sort_values("article_count", ascending=False)
)

obligors_with_zero = obligors_df[~obligors_df["id"].isin(links_df["obligor_id"])]
print(f"Obligors with 0 linked articles: {len(obligors_with_zero)}")
if len(obligors_with_zero):
    print(obligors_with_zero[["name", "ticker"]].to_string(index=False))

top20 = article_counts.head(20)
fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top20["ticker"].fillna(top20["name"]), top20["article_count"], color="steelblue")
ax.set_xlabel("Article count")
ax.set_title("Top 20 obligors by linked article count")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 4 — Entity extraction quality
import json

with engine.connect() as conn:
    org_count = conn.execute(
        text("SELECT COUNT(*) FROM processed_articles WHERE entities::text LIKE '%\"ORG\"%'")
    ).scalar()

mapped_processed = links_df["article_id"].nunique()
pct_mapped = mapped_processed / processed_count * 100 if processed_count else 0

print("=== Entity Extraction Quality ===")
print(f"Processed articles with ORG entities: {org_count:,}")
print(f"Processed articles mapped to an obligor: {mapped_processed:,} ({pct_mapped:.1f}%)")
print()
print("Top 10 obligors by article count:")
print(article_counts[["name", "ticker", "article_count"]].head(10).to_string(index=False))

In [ ]:
# Cell 5 — Timeline chart
articles_df["pub_date"] = articles_df["published_at"].dt.date
articles_per_day = articles_df.groupby("pub_date").size().rename("article_count")

links_with_date = links_df.merge(
    articles_df[["id", "pub_date"]], left_on="article_id", right_on="id"
)
links_per_day = links_with_date.groupby("pub_date")["article_id"].nunique().rename("link_count")

timeline = pd.DataFrame({"articles": articles_per_day, "linked": links_per_day}).fillna(0)
timeline = timeline.sort_index()

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(timeline))
ax.bar(x, timeline["articles"], label="Total articles", color="steelblue", alpha=0.7)
ax.bar(x, timeline["linked"], label="Linked to obligor", color="darkorange", alpha=0.9)
ax.set_xticks(list(x))
ax.set_xticklabels([str(d) for d in timeline.index], rotation=45, ha="right")
ax.set_ylabel("Article count")
ax.set_title("Articles published per day — total vs. obligor-linked")
ax.legend()
plt.tight_layout()
plt.show()